# Regresión Logística — FIFA 17 + FIFA 18

Clasificación binaria para predecir si un jugador es **élite** (Overall ≥ 80).

| Parámetro | Valor |
|-----------|-------|
| **Dataset** | FIFA 17 + FIFA 18 combinados |
| **Muestras (m)** | 35,487 jugadores |
| **Features (n)** | 40 variables numéricas |
| **Target** | `élite` → 1 si Overall ≥ 80, 0 si no |
| **Split** | 80% entrenamiento / 20% prueba (sin solapamiento) |

**Modelos implementados:** Descenso por Gradiente · Optimización TNC (scipy)

## 1. Librerías y Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns
%matplotlib inline
print('✓ Librerías cargadas')

## 2. Cargar Dataset

Se carga el CSV desde Google Drive, se seleccionan las 40 features numéricas y se crea la variable objetivo binaria: `élite = 1` si `Overall ≥ 80`, `0` en caso contrario.

In [ ]:
RUTA = '/content/drive/MyDrive/Universida/IA/Dataset/Clasificacion/FIFA_17_18_combined.csv'
df   = pd.read_csv(RUTA)

features = [
    'Age','Potential','Crossing','Finishing','HeadingAccuracy',
    'ShortPassing','Volleys','Dribbling','Curve','FKAccuracy',
    'LongPassing','BallControl','Acceleration','SprintSpeed',
    'Agility','Reactions','Balance','ShotPower','Jumping',
    'Stamina','Strength','LongShots','Aggression','Interceptions',
    'Positioning','Vision','Penalties','Composure','Marking',
    'StandingTackle','SlidingTackle','GKDiving','GKHandling',
    'GKKicking','GKPositioning','GKReflexes',
    'International Reputation','Weak Foot','Skill Moves','FIFA_Year'
]

df_clean = df[features + ['Overall']].dropna()
X = df_clean[features].values.astype(float)
y = (df_clean['Overall'].values >= 80).astype(int)

print(f'✓ Dataset cargado: {X.shape[0]:,} jugadores, {X.shape[1]} features')
print(f'  Élite (y=1): {y.sum():,} jugadores ({y.mean()*100:.1f}%)')
print(f'  No élite (y=0): {(1-y).sum():,} jugadores ({(1-y).mean()*100:.1f}%)')
df_clean.head(3)

## 3. División Train / Test — 80% / 20%

Se dividen los datos **aleatoriamente** con semilla fija para reproducibilidad. Los datos de prueba quedan completamente aislados: **no participan en ningún paso del entrenamiento** (normalización, ajuste de θ ni selección de hiperparámetros).

In [ ]:
np.random.seed(42)
m_total  = X.shape[0]
idx      = np.random.permutation(m_total)       # mezclar índices
corte    = int(0.8 * m_total)                   # 80% train

idx_train, idx_test = idx[:corte], idx[corte:]  # sin solapamiento

X_train, y_train = X[idx_train], y[idx_train]
X_test,  y_test  = X[idx_test],  y[idx_test]

print(f'✓ Split completado (semilla=42)')
print(f'  Train : {X_train.shape[0]:,} muestras ({X_train.shape[0]/m_total*100:.0f}%)')
print(f'  Test  : {X_test.shape[0]:,} muestras  ({X_test.shape[0]/m_total*100:.0f}%)')
print(f'  Élite en train: {y_train.mean()*100:.1f}%  |  Élite en test: {y_test.mean()*100:.1f}%')
print('  ⚠ Los datos de test NO se usan en ningún paso del entrenamiento')

## 4. Normalización

$$x_j := \\frac{x_j - \\mu_j}{\\sigma_j}$$

**Regla crítica:** `μ` y `σ` se calculan **únicamente** sobre el conjunto de entrenamiento. Luego se aplican al conjunto de prueba. Esto evita *data leakage* (filtración de información del test al modelo).

In [ ]:
# μ y σ calculados SOLO sobre train
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0) + 1e-8

# Normalizar con los parámetros del train
X_train_n = (X_train - X_mean) / X_std
X_test_n  = (X_test  - X_mean) / X_std   # mismos μ y σ, sin recalcular

# Agregar columna de unos (término de intercepción θ₀)
X_train_b = np.hstack([np.ones((X_train_n.shape[0], 1)), X_train_n])
X_test_b  = np.hstack([np.ones((X_test_n.shape[0],  1)), X_test_n])

n = X_train_n.shape[1]
print(f'✓ Normalización aplicada (sin data leakage)')
print(f'  X_train_b: {X_train_b.shape}  |  X_test_b: {X_test_b.shape}')

## 5. Visualización — Reactions vs Potential

Se grafican las dos features con mayor correlación con el target para visualizar la separabilidad de clases en el conjunto de entrenamiento.

In [ ]:
r_idx = features.index('Reactions')
p_idx = features.index('Potential')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (X_p, y_p, titulo) in zip(axes, [
        (X_train, y_train, 'Train (80%)'),
        (X_test,  y_test,  'Test (20%)')]):
    ax.scatter(X_p[y_p==0, r_idx], X_p[y_p==0, p_idx],
               c='gold', edgecolors='k', s=10, alpha=0.3, label='No Élite')
    ax.scatter(X_p[y_p==1, r_idx], X_p[y_p==1, p_idx],
               c='steelblue', edgecolors='k', s=15, alpha=0.6, label='Élite')
    ax.set_xlabel('Reactions'); ax.set_ylabel('Potential')
    ax.set_title(f'FIFA — {titulo}'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Función Sigmoide

Convierte cualquier valor real en una probabilidad [0, 1]:

$$g(z) = \\frac{1}{1 + e^{-z}}$$

La hipótesis del modelo es: $h_\\theta(x) = g(\\theta^T x)$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Prueba
print('Prueba función sigmoide:')
for z in [-10, -1, 0, 1, 10]:
    print(f'  sigmoid({z:4}) = {sigmoid(z):.6f}')

# Gráfica
z_r = np.linspace(-10, 10, 300)
plt.figure(figsize=(8, 3))
plt.plot(z_r, sigmoid(z_r), 'b-', lw=2)
plt.axhline(0.5, color='r', ls='--', alpha=0.7, label='Umbral = 0.5')
plt.axvline(0,   color='gray', ls='--', alpha=0.5)
plt.title('Función Sigmoide'); plt.xlabel('z'); plt.ylabel('g(z)')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 7. Función de Costo (Binary Cross-Entropy)

$$J(\\theta) = -\\frac{1}{m}\\sum_{i=1}^{m}\\left[ y^{(i)}\\log(h_\\theta(x^{(i)})) + (1-y^{(i)})\\log(1-h_\\theta(x^{(i)})) \\right]$$

Se aplica `clip` para evitar `log(0)`.

In [ ]:
def calcularCosto(theta, X, y):
    h = np.clip(sigmoid(X @ theta), 1e-10, 1-1e-10)
    return -(1/len(y)) * np.sum(y*np.log(h) + (1-y)*np.log(1-h))

theta0 = np.zeros(n + 1)
print(f'✓ Costo con theta=0 (train): {calcularCosto(theta0, X_train_b, y_train):.4f}')
print(f'  Esperado aprox.: 0.6931  (log(2) para clases balanceadas)')

## 8. Modelo 1 — Descenso por el Gradiente

Entrenado **únicamente con datos de train**. Se registra el costo y la precisión por iteración.

$$\\theta := \\theta - \\frac{\\alpha}{m} X^T(h_\\theta(X) - y)$$

In [ ]:
def descensoGradiente(theta, X, y, alpha, iters):
    theta, J_hist, acc_hist = theta.copy(), [], []
    for _ in range(iters):
        h     = sigmoid(X @ theta)
        theta = theta - (alpha/len(y)) * X.T @ (h - y)
        J_hist.append(calcularCosto(theta, X, y))
        acc_hist.append(np.mean((h >= 0.5).astype(int) == y) * 100)
    return theta, J_hist, acc_hist

theta_gd, J_hist, acc_hist = descensoGradiente(np.zeros(n+1), X_train_b, y_train, alpha=0.1, iters=300)

# Gráficas de convergencia
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(J_hist, color='steelblue', lw=2)
axes[0].set(title='Costo — Gradiente Descendente', xlabel='Iteraciones', ylabel='J(θ)')
axes[0].grid(True, alpha=0.3)
axes[1].plot(acc_hist, color='darkorange', lw=2)
axes[1].set(title='Precisión en Train — Gradiente', xlabel='Iteraciones', ylabel='Precisión (%)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'✓ Entrenamiento completado (solo con datos train)')
print(f'  Costo inicial : {J_hist[0]:.4f}')
print(f'  Costo final   : {J_hist[-1]:.4f}')
print(f'  Precisión final en train: {acc_hist[-1]:.2f}%')

## 9. Modelo 2 — Optimización TNC (scipy)

Método de Newton Truncado. Calcula el gradiente analítico junto con el costo, logrando convergencia más rápida y precisa que el gradiente descendente básico. Entrenado también **solo con datos de train**.

In [ ]:
def costFunction(theta, X, y):
    h    = np.clip(sigmoid(X @ theta), 1e-10, 1-1e-10)
    J    = -(1/len(y)) * np.sum(y*np.log(h) + (1-y)*np.log(1-h))
    grad = (1/len(y)) * X.T @ (h - y)
    return J, grad

res = optimize.minimize(costFunction, np.zeros(n+1), (X_train_b, y_train),
                        jac=True, method='TNC', options={'maxiter':1000})
theta_opt, cost_opt = res.x, res.fun

print(f'✓ Optimización TNC completada (solo con datos train)')
print(f'  Costo final (TNC)       : {cost_opt:.4f}')
print(f'  Costo final (Gradiente) : {J_hist[-1]:.4f}')
print(f'  Primeros 5 θ: {theta_opt[:5].round(4)}')

## 10. Evaluación — Train vs Test

Se evalúan **ambos modelos** sobre el conjunto de **prueba (20%)** que nunca fue visto durante el entrenamiento. Esto mide la capacidad real de generalización del modelo.

> Un modelo que tiene alta precisión en train pero baja en test sufre **overfitting**.

In [ ]:
def evaluar(theta, X, y, nombre, split):
    pred = (sigmoid(X @ theta) >= 0.5).astype(int)
    acc  = np.mean(pred == y) * 100
    costo = calcularCosto(theta, X, y)
    print(f'  [{nombre}] {split:<6} → Precisión: {acc:.2f}%  |  Costo: {costo:.4f}')
    return pred, acc

print('── Descenso por Gradiente ──')
_, acc_gd_train = evaluar(theta_gd,  X_train_b, y_train, 'Gradiente', 'TRAIN')
p_gd_test, acc_gd_test   = evaluar(theta_gd,  X_test_b,  y_test,  'Gradiente', 'TEST ')

print('── Optimización TNC ────────')
_, acc_tnc_train = evaluar(theta_opt, X_train_b, y_train, 'TNC',       'TRAIN')
p_tnc_test, acc_tnc_test = evaluar(theta_opt, X_test_b,  y_test,  'TNC',       'TEST ')

print()
print(f'  Diferencia Gradiente (train-test): {acc_gd_train - acc_gd_test:.2f}%')
print(f'  Diferencia TNC       (train-test): {acc_tnc_train - acc_tnc_test:.2f}%')
print('  → Diferencias pequeñas indican buena generalización (sin overfitting)')

## 11. Matrices de Confusión — Conjunto de Prueba

Se evalúa el desempeño detallado de cada modelo sobre los datos de **test**:

- **VP** (Verdadero Positivo): élite predicho como élite
- **VN** (Verdadero Negativo): no élite predicho como no élite
- **FP** (Falso Positivo): no élite predicho erróneamente como élite
- **FN** (Falso Negativo): élite no detectado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (pred, titulo) in zip(axes, [
        (p_gd_test,  'Gradiente Descendente'),
        (p_tnc_test, 'Optimización TNC')]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Élite','Élite'],
                yticklabels=['No Élite','Élite'])
    ax.set_title(f'Matriz de Confusión — {titulo}\n(Test 20%)', fontsize=12)
    ax.set_ylabel('Real'); ax.set_xlabel('Predicho')
plt.tight_layout(); plt.show()

print('── Reporte detallado — TNC (Test) ──')
print(classification_report(y_test, p_tnc_test, target_names=['No Élite','Élite']))

## 12. Curva ROC y AUC — Conjunto de Prueba

La curva ROC muestra el balance entre **sensibilidad** (TPR) y **especificidad** (1-FPR) para todos los umbrales posibles.

**AUC (Área bajo la curva):** 1.0 = perfecto · 0.5 = aleatorio · Se calcula sobre el **test set**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for theta, color, nombre in [
        (theta_gd,  'darkorange', 'Gradiente'),
        (theta_opt, 'steelblue',  'TNC')]:
    y_prob = sigmoid(X_test_b @ theta)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC={roc_auc:.3f})')
    print(f'  AUC {nombre}: {roc_auc:.4f}')
ax.plot([0,1],[0,1],'r--',lw=1.5, label='Aleatorio (AUC=0.5)')
ax.set(xlabel='Falsos Positivos', ylabel='Verdaderos Positivos',
       title='Curva ROC — Test Set (20%)')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3); plt.show()

## 13. Features más Importantes

Los valores absolutos de los coeficientes θ indican el peso de cada feature en la decisión del modelo. Un coeficiente mayor implica mayor influencia sobre la probabilidad de ser clasificado como élite.

In [ ]:
importancias = pd.Series(np.abs(theta_opt[1:]), index=features).sort_values(ascending=True).tail(15)
plt.figure(figsize=(9, 6))
importancias.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Features — Modelo TNC', fontsize=13)
plt.xlabel('|Coeficiente θ|'); plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
print('Top 10 features:')
print(importancias.sort_values(ascending=False).head(10).round(4).to_string())

## 14. Predicción para un Jugador Específico

Se puede ingresar un jugador nuevo con sus estadísticas y obtener la probabilidad de ser élite. El vector se normaliza con los mismos `μ`/`σ` del entrenamiento.

In [ ]:
def predecir_jugador(stats, features, X_mean, X_std, theta):
    x      = np.array([stats[f] for f in features])
    x_norm = (x - X_mean) / X_std
    prob   = sigmoid(np.r_[1, x_norm] @ theta)
    clase  = 'ÉLITE ⭐' if prob >= 0.5 else 'No Élite'
    print(f'  Probabilidad de ser élite: {prob*100:.1f}%  →  {clase}')
    return prob

jugador_bueno = dict(zip(features,
    [27,88,80,85,78,83,79,85,80,75,78,86,82,80,83,85,80,82,
     75,82,78,80,72,74,84,82,78,85,70,72,68,12,11,13,12,11,4,3,3,2018]))
jugador_malo  = dict(zip(features,
    [19,62,45,40,38,50,35,48,42,40,44,50,65,67,60,55,62,45,
     60,68,65,38,55,50,42,48,40,45,52,50,48,10,10,10,10,10,1,2,1,2017]))

print('Jugador con stats altas:')
predecir_jugador(jugador_bueno, features, X_mean, X_std, theta_opt)
print('Jugador con stats bajas:')
predecir_jugador(jugador_malo, features, X_mean, X_std, theta_opt)

## 15. Resumen Final

Comparación completa de los dos modelos evaluados sobre el **conjunto de prueba (20%)**.

In [ ]:
y_prob_gd  = sigmoid(X_test_b @ theta_gd)
y_prob_tnc = sigmoid(X_test_b @ theta_opt)
auc_gd  = auc(*roc_curve(y_test, y_prob_gd)[:2])
auc_tnc = auc(*roc_curve(y_test, y_prob_tnc)[:2])

print('='*62)
print('       RESUMEN — REGRESIÓN LOGÍSTICA FIFA 17+18')
print('='*62)
print(f'  Dataset        : FIFA 17 + FIFA 18 combinados')
print(f'  Total muestras : {len(y):,}  →  Train: {len(y_train):,}  |  Test: {len(y_test):,}')
print(f'  Features (n)   : {n}')
print(f'  Élite en test  : {y_test.sum():,} ({y_test.mean()*100:.1f}%)')
print('-'*62)
print(f'  {"Métrica":<28} {"Gradiente":>12} {"TNC":>12}')
print(f'  {"-"*56}')
print(f'  {"Precisión TRAIN":<28} {acc_gd_train:>11.2f}% {acc_tnc_train:>11.2f}%')
print(f'  {"Precisión TEST":<28} {acc_gd_test:>11.2f}% {acc_tnc_test:>11.2f}%')
print(f'  {"Costo final (train)":<28} {J_hist[-1]:>12.4f} {cost_opt:>12.4f}')
print(f'  {"AUC (test)":<28} {auc_gd:>12.4f} {auc_tnc:>12.4f}')
print('='*62)